<a href="https://colab.research.google.com/github/nikhilmooloo02-stack/CLARIX_AI_AGENT/blob/main/CLARIX_AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Cell 1 — Install packages
!pip install anthropic gradio chromadb reportlab PyPDF2 -q
print("Packages ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.7 MB/s eta 0:00:00
Packages ready


In [24]:
# Cell 2 — CLARIX Core
import anthropic
import chromadb
import re
from google.colab import userdata

# CONNECTION
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# PRODUCT KNOWLEDGE BASE
chroma_client = chromadb.Client()
knowledge_base = chroma_client.create_collection(
name="shaneal_products",
get_or_create=True
)

products = """
PAPER:
- Butterfly A4 Paper 80gsm 500 sheets - R89.00
- Rotatrim A4 Paper 75gsm 500 sheets - R75.00
- A3 Paper 80gsm 500 sheets - R165.00
- Letterhead Paper A4 100 sheets - R45.00

PENS AND WRITING:
- Bic Ballpoint Pens Blue Box of 50 - R120.00
- Pilot G2 Gel Pens Black Pack of 12 - R95.00
- Staedtler Permanent Markers Pack of 10 - R85.00
- Highlighters Assorted Pack of 5 - R55.00

PPE:
- Surgical Face Masks Box of 50 - R95.00
- Nitrile Gloves Box of 100 - R180.00
- Safety Goggles - R45.00
- Reflective Safety Vest - R120.00

OFFICE EQUIPMENT:
- Bantex A4 Lever Arch File - R45.00
- Stapler Heavy Duty - R135.00
- Calculator Scientific - R250.00
- Whiteboard A1 - R850.00
- Shredder 10 Sheet - R1200.00

HOUSEHOLD CONSUMABLES:
- Refuse Bags Black Roll of 20 - R35.00
- Hand Sanitiser 500ml - R65.00
- Multipurpose Cleaning Spray 750ml - R45.00
- Toilet Paper 9 Roll Pack - R55.00
"""

knowledge_base.add(
documents=[products],
ids=["shaneal_product_list"]
)

# SYSTEM PROMPT
SYSTEM_PROMPT = """You are CLARIX, the professional AI assistant for
ShaNeal Distributors — a stationery, PPE, office equipment and household
consumables distributor based in Pretoria, South Africa.

You serve retail customers and bulk business and school clients.

PRODUCTS AND PRICING:
PAPER: Butterfly A4 Paper 80gsm 500 sheets R89.00 | Rotatrim A4 Paper
75gsm 500 sheets R75.00 | A3 Paper 80gsm 500 sheets R165.00 |
Letterhead Paper A4 100 sheets R45.00
PENS: Bic Ballpoint Pens Blue Box of 50 R120.00 | Pilot G2 Gel Pens
Black Pack of 12 R95.00 | Staedtler Permanent Markers Pack of 10 R85.00
| Highlighters Assorted Pack of 5 R55.00
PPE: Surgical Face Masks Box of 50 R95.00 | Nitrile Gloves Box of 100
R180.00 | Safety Goggles R45.00 | Reflective Safety Vest R120.00
OFFICE EQUIPMENT: Bantex A4 Lever Arch File R45.00 | Stapler Heavy Duty
R135.00 | Calculator Scientific R250.00 | Whiteboard A1 R850.00 |
Shredder 10 Sheet R1200.00
HOUSEHOLD CONSUMABLES: Refuse Bags Black Roll of 20 R35.00 | Hand
Sanitiser 500ml R65.00 | Multipurpose Cleaning Spray 750ml R45.00 |
Toilet Paper 9 Roll Pack R55.00

CONTACT INFORMATION:
- Phone: 070 070 0770
- Email: ShaNeal@lantic.co.za
- Address: 332 Paul Kruger Street, Corner Van Heerden Street,
Capital Park, Pretoria 0084
- Website: www.ShaNealonline.co.za
- Business Hours: Monday to Friday, 8am to 6pm

BEHAVIOUR:
- Recommend specific products with prices
- Cross-sell related products naturally
- For bulk orders ask for organisation name and delivery address
- For complaints apologise sincerely and offer a clear solution
- For orders over R10,000 escalate to a human consultant
- When customers ask for contact details share all of the above
- Always be warm, professional and solution-focused"""

print("CLARIX is ready")

CLARIX is ready


In [27]:
# Cell 3 — CLARIX Complete UI
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Table, TableStyle,
                                Paragraph, Spacer)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
import gradio as gr
import datetime
import random
import re
import base64

WHATSAPP_NUMBER = "27723304651"

def generate_quote(customer_name, company_name, items_text):
    price_list = {
        "butterfly a4 paper": 89.00,
        "rotatrim a4 paper": 75.00,
        "a3 paper": 165.00,
        "letterhead paper": 45.00,
        "bic ballpoint pens": 120.00,
        "pilot g2 gel pens": 95.00,
        "staedtler permanent markers": 85.00,
        "highlighters": 55.00,
        "surgical face masks": 95.00,
        "nitrile gloves": 180.00,
        "safety goggles": 45.00,
        "reflective safety vest": 120.00,
        "bantex a4 lever arch file": 45.00,
        "stapler heavy duty": 135.00,
        "calculator scientific": 250.00,
        "whiteboard a1": 850.00,
        "shredder 10 sheet": 1200.00,
        "refuse bags": 35.00,
        "hand sanitiser": 65.00,
        "multipurpose cleaning spray": 45.00,
        "toilet paper": 55.00,
    }

    line_items = []
    for line in items_text.strip().split('\n'):
        if ',' in line:
            parts = line.split(',')
            product = parts[0].strip()
            try:
                qty = int(parts[1].strip())
            except:
                qty = 1
            price = None
            for key in price_list:
                if key in product.lower():
                    price = price_list[key]
                    break
            if price is None:
                price = 0.00
            line_items.append([product, qty, price, price * qty])

    if not line_items:
        return None, "No valid items found."

    subtotal = sum(item[3] for item in line_items)
    vat = subtotal * 0.15
    total = subtotal + vat
    quote_num = (f"SND-{random.randint(1000,9999)}"
                 f"-{datetime.datetime.now().year}")
    today = datetime.datetime.now().strftime("%d %B %Y")
    valid_until = (datetime.datetime.now() +
                   datetime.timedelta(days=30)).strftime("%d %B %Y")

    filename = f"/content/CLARIX_Quote_{quote_num}.pdf"
    doc = SimpleDocTemplate(filename, pagesize=A4,
                            rightMargin=20*mm, leftMargin=20*mm,
                            topMargin=20*mm, bottomMargin=20*mm)
    styles = getSampleStyleSheet()
    elements = []

    header_style = ParagraphStyle(
        'Header', parent=styles['Normal'],
        fontSize=24, textColor=colors.HexColor('#1A1A1A'),
        spaceAfter=2*mm, fontName='Helvetica-Bold')
    sub_style = ParagraphStyle(
        'Sub', parent=styles['Normal'],
        fontSize=10, textColor=colors.HexColor('#C9A84C'),
        spaceAfter=1*mm)
    normal_style = ParagraphStyle(
        'Normal2', parent=styles['Normal'],
        fontSize=9, textColor=colors.HexColor('#2C2C2C'),
        spaceAfter=1*mm)

    elements.append(Paragraph("SHANEAL DISTRIBUTORS", header_style))
    elements.append(Paragraph(
        "Stationery · PPE · Office Equipment · Household Consumables",
        sub_style))
    elements.append(Spacer(1, 3*mm))

    divider = Table([['']], colWidths=[170*mm])
    divider.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#C9A84C')),
        ('ROWHEIGHTS', (0,0), (-1,-1), 2),
    ]))
    elements.append(divider)
    elements.append(Spacer(1, 5*mm))

    info_data = [
        [Paragraph('<b>QUOTATION</b>', styles['Normal']),
         Paragraph('<b>Bill To:</b>', styles['Normal'])],
        [Paragraph(f'Quote No: {quote_num}', normal_style),
         Paragraph(customer_name, normal_style)],
        [Paragraph(f'Date: {today}', normal_style),
         Paragraph(company_name, normal_style)],
        [Paragraph(f'Valid Until: {valid_until}', normal_style),
         Paragraph('', normal_style)],
    ]
    info_table = Table(info_data, colWidths=[85*mm, 85*mm])
    info_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'TOP')
    ]))
    elements.append(info_table)
    elements.append(Spacer(1, 8*mm))

    table_data = [['DESCRIPTION', 'QTY', 'UNIT PRICE', 'TOTAL']]
    for item in line_items:
        table_data.append([
            item[0], str(item[1]),
            f"R {item[2]:,.2f}", f"R {item[3]:,.2f}"
        ])
    table_data.append(['', '', 'Subtotal:', f"R {subtotal:,.2f}"])
    table_data.append(['', '', 'VAT (15%):', f"R {vat:,.2f}"])
    table_data.append(['', '', 'TOTAL DUE:', f"R {total:,.2f}"])

    items_table = Table(table_data,
                        colWidths=[90*mm, 20*mm, 30*mm, 30*mm])
    items_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1A1A1A')),
        ('TEXTCOLOR', (0,0), (-1,0), colors.HexColor('#C9A84C')),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('ALIGN', (1,0), (-1,-1), 'RIGHT'),
        ('ROWBACKGROUNDS', (0,1), (-1,-4),
         [colors.HexColor('#FAFAF8'), colors.white]),
        ('FONTNAME', (2,-1), (-1,-1), 'Helvetica-Bold'),
        ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor('#1A1A1A')),
        ('TEXTCOLOR', (0,-1), (-1,-1), colors.HexColor('#C9A84C')),
        ('LINEABOVE', (0,-3), (-1,-3), 0.5,
         colors.HexColor('#E8E0D0')),
        ('GRID', (0,0), (-1,-4), 0.25, colors.HexColor('#E8E0D0')),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
        ('RIGHTPADDING', (0,0), (-1,-1), 6),
    ]))
    elements.append(items_table)
    elements.append(Spacer(1, 10*mm))

    footer_data = [[Paragraph(
        'ShaNeal Distributors | 332 Paul Kruger Street, '
        'Capital Park, Pretoria 0084<br/>'
        'Tel: 070 070 0770 | Email: ShaNeal@lantic.co.za | '
        'www.ShaNealonline.co.za',
        ParagraphStyle('Footer', parent=styles['Normal'],
                       fontSize=8,
                       textColor=colors.HexColor('#888888'))
    )]]
    footer_table = Table(footer_data, colWidths=[170*mm])
    footer_table.setStyle(TableStyle([
        ('LINEABOVE', (0,0), (-1,0), 0.5, colors.HexColor('#C9A84C')),
        ('TOPPADDING', (0,0), (-1,-1), 4),
    ]))
    elements.append(footer_table)
    doc.build(elements)
    return filename, quote_num

def parse_and_generate_quote(response_text):
    if "GENERATE_QUOTE" not in response_text:
        return response_text
    try:
        name_match = re.search(r'NAME:\s*(.+)', response_text)
        company_match = re.search(r'COMPANY:\s*(.+)', response_text)
        items_match = re.search(r'ITEMS:\n(.*?)END_QUOTE',
                                response_text, re.DOTALL)
        if not all([name_match, company_match, items_match]):
            return response_text
        customer_name = name_match.group(1).strip()
        company_name = company_match.group(1).strip()
        items_text = items_match.group(1).strip()
        filename, quote_num = generate_quote(
            customer_name, company_name, items_text)

        # Convert PDF to base64 download link
        with open(filename, 'rb') as f:
            pdf_bytes = f.read()
            b64 = base64.b64encode(pdf_bytes).decode('utf-8')
            download_link = f'data:application/pdf;base64,{b64}'

        return f"""✅ Quote {quote_num} generated successfully!\n\n**Customer:** {customer_name} — {company_name}\n\n📄 **[Click here to download your quote PDF]({download_link})**\n\nValid for 30 days. Is there anything else I can help you with?"""
    except Exception as e:
        return f"I had trouble generating the quote: {str(e)}"

def check_escalation(response_text):
    return "ESCALATE_TO_HUMAN" in response_text

def get_whatsapp_link(summary):
    message = (f"Hello ShaNeal Distributors, "
               f"I need assistance. {summary}")
    encoded = message.replace(' ', '%20').replace('\n', '%0A')
    return f"https://wa.me/{WHATSAPP_NUMBER}?text={encoded}"

def chat(message, history):
    results = knowledge_base.query(
        query_texts=[message], n_results=1)
    product_context = ""
    if results['documents'][0]:
        product_context = results['documents'][0][0]

    enhanced_message = f"""Customer query: {message}
ShaNeal product information: {product_context}"""

    conversation = []
    for item in history:
        conversation.append({"role": "user", "content": item[0]})
        conversation.append({"role": "assistant",
                             "content": item[1]})
    conversation.append({"role": "user",
                         "content": enhanced_message})

    full_prompt = SYSTEM_PROMPT + """

FINANCIAL ANALYSIS:
When customers ask about VAT or cost calculations:
- Calculate VAT at 15% showing subtotal, VAT amount and total
- For bulk orders calculate the full total cost
- Always show amounts in South African Rand (R)
- Give clear itemised breakdowns
- Recommend cost saving options where relevant

QUOTE GENERATION:
When a customer asks for a quote or invoice:
1. Ask for their name and company name if not provided
2. Confirm the products and quantities
3. Respond with EXACTLY this format:

GENERATE_QUOTE
NAME: [customer name]
COMPANY: [company name]
ITEMS:
[product name], [quantity]
[product name], [quantity]
END_QUOTE

HUMAN ESCALATION:
Include ESCALATE_TO_HUMAN when:
- Customer requests a human agent
- Order likely exceeds R10,000
- Customer is very angry or mentions legal action
- Issue is too complex to resolve"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        system=full_prompt,
        messages=conversation
    )

    reply = response.content[0].text
    display_reply = parse_and_generate_quote(reply)

    if check_escalation(display_reply):
        summary = f"Customer issue: {message}"
        whatsapp_link = get_whatsapp_link(summary)
        clean_reply = display_reply.replace(
            "ESCALATE_TO_HUMAN", "").strip()
        display_reply = f"""{clean_reply}\n\n---\n🔴 **This issue requires a human agent.**\n\n👉 [**Click here to chat with a ShaNeal agent on WhatsApp**]({whatsapp_link})\n\nOur team is available **Monday to Friday, 8am to 6pm**.\n📞 You can also call us directly: **070 070 0770**"""

    return display_reply

gr.ChatInterface(
    fn=chat,
    title="CLARIX — ShaNeal Distributors AI Assistant",
    description="Ask about products, request a quote, or get support.",
    examples=[
        "Can I get a quote for 10 reams of A4 paper?",
        "Calculate VAT on an invoice of R5,500.",
        "What is the total cost of PPE for 50 staff?",
        "I need PPE for 20 staff members.",
        "What stationery products do you stock?",
        "Help me set up a new office for 10 staff.",
        "My order arrived damaged, I need a refund urgently.",
        "I need to speak to a human agent.",
        "What are your business hours?",
        "How can I contact ShaNeal Distributors?",
    ],
    theme=gr.themes.Soft()
).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0f253669f62b349e5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
